In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, upper, round, concat, lit # Besoin pour Q11 et +
from pyspark.sql.types import DateType, DoubleType, IntegerType, StringType # besoin pour la Q13 et les casts

In [2]:
# Initialisation de la session Spark
spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

print(spark.version);

# Chemin relatif vers les CSV
PATH = "../data/"

# Création des DataFrames
# inferSchema=True permet de déduire les types de chaque colonne automatiquement
df_categories = spark.read.csv(f"{PATH}categories.csv", header=True, inferSchema=True);
df_customers = spark.read.csv(f"{PATH}customers.csv", header=True, inferSchema=True);
df_employees = spark.read.csv(f"{PATH}employees.csv", header=True, inferSchema=True);
df_orders_details = spark.read.csv(f"{PATH}order_details.csv", header=True, inferSchema=True);
df_orders = spark.read.csv(f"{PATH}orders.csv", header=True, inferSchema=True);
df_products = spark.read.csv(f"{PATH}products.csv", header=True, inferSchema=True);
df_shippers = spark.read.csv(f"{PATH}shippers.csv", header=True, inferSchema=True);
df_suppliers = spark.read.csv(f"{PATH}suppliers.csv", header=True, inferSchema=True);

# Dictionnaire de tout les DataFrames
df_collection = {"categories" : df_categories, 
                 "clients" : df_customers, 
                 "employees" : df_employees, 
                 "details_commandes" : df_orders_details, 
                 "commandes" : df_orders, 
                 "produits" : df_products, 
                 "transporteurs" : df_shippers, 
                 "fournisseurs": df_suppliers}

4.2.0


# Q11 — Valeurs nulles
Pour chaque DataFrame, compter le nombre de valeurs nulles par colonne. Utiliser une boucle et
df.filter(col(c).isNull()).count().

In [3]:
for name, df in df_collection.items():
    print(f"DataFrame: {name}")
    for c in df.columns:
        # Compter les valeurs nulles dans chaque colonne
        null_count = df.filter(col(c).isNull()).count()
        # N'afficher que les colonnes avec des valeurs nulles
        if null_count > 0:
            print(f"Nombre de valeurs nulles dans la colonne {c} : {null_count}")
    print("\n")

DataFrame: categories
Nombre de valeurs nulles dans la colonne picture : 8


DataFrame: clients
Nombre de valeurs nulles dans la colonne region : 60
Nombre de valeurs nulles dans la colonne postal_code : 1
Nombre de valeurs nulles dans la colonne fax : 22


DataFrame: employees
Nombre de valeurs nulles dans la colonne region : 4
Nombre de valeurs nulles dans la colonne photo : 9
Nombre de valeurs nulles dans la colonne reports_to : 1


DataFrame: details_commandes


DataFrame: commandes
Nombre de valeurs nulles dans la colonne shipped_date : 21
Nombre de valeurs nulles dans la colonne ship_region : 507
Nombre de valeurs nulles dans la colonne ship_postal_code : 19


DataFrame: produits


DataFrame: transporteurs


DataFrame: fournisseurs
Nombre de valeurs nulles dans la colonne region : 20
Nombre de valeurs nulles dans la colonne fax : 16
Nombre de valeurs nulles dans la colonne homepage : 24




# Q12 — Supprimer les nulls
Dans df_orders, supprimer les lignes où shipped_date est null (commandes non livrées). Dans df_products,
remplacer les valeurs nulles de unit_price par la médiane.

In [4]:
nb = df_orders.count()

# Suppression des lignes avec shipped_date null dans df_orders
df_orders = df_orders.dropna(subset=["shipped_date"])

print("Nombre de ligne avant suppression des nulls dans shipped_date : ", nb)
print("Nombre de lignes après suppression des nulls dans shipped_date : ", df_orders.count())

Nombre de ligne avant suppression des nulls dans shipped_date :  830
Nombre de lignes après suppression des nulls dans shipped_date :  809


In [5]:
# Replacement des valeurs nulles de Produits (meme si il n'y en a pas dans notre jeu de données)*

# Calcul de la médiane sur les valeurs non nulles de unit_price
# Le 0.0 est pour la précision de l'estimation (entre 0 et 1), plus c'est proche de 0, plus c'est précis
# Le [0] est pour prendre la premiere valeur de la liste retournée par approxQuantile
unit_price_median = df_products.na.drop().approxQuantile("unit_price", [0.5], 0.0)[0]

# Remplacement des valeurs nulles de unit_price par la médiane
df_products = df_products.fillna({"unit_price": unit_price_median})

 # Q13 — Cast des types
Dans df_orders, caster order_date, required_date et shipped_date en type DateType. Dans df_order_details,
caster unit_price en DoubleType et quantity en IntegerType.

In [6]:
print(df_orders.dtypes)
print(df_orders_details.dtypes)

[('order_id', 'int'), ('customer_id', 'string'), ('employee_id', 'int'), ('order_date', 'date'), ('required_date', 'date'), ('shipped_date', 'date'), ('ship_via', 'int'), ('freight', 'double'), ('ship_name', 'string'), ('ship_address', 'string'), ('ship_city', 'string'), ('ship_region', 'string'), ('ship_postal_code', 'string'), ('ship_country', 'string')]
[('order_id', 'int'), ('product_id', 'int'), ('unit_price', 'double'), ('quantity', 'int'), ('discount', 'double')]


In [7]:
df_orders = df_orders.withColumn("order_date", col("order_date").cast(DateType()));
df_orders = df_orders.withColumn("required_date", col("required_date").cast(DateType()));
df_orders = df_orders.withColumn("shipped_date", col("shipped_date").cast(DateType()));

df_orders_details = df_orders_details.withColumn("unit_price", col("unit_price").cast(DoubleType()));
df_orders_details = df_orders_details.withColumn("quantity", col("quantity").cast(IntegerType()));

In [8]:
print(df_orders.dtypes)
print(df_orders_details.dtypes)

[('order_id', 'int'), ('customer_id', 'string'), ('employee_id', 'int'), ('order_date', 'date'), ('required_date', 'date'), ('shipped_date', 'date'), ('ship_via', 'int'), ('freight', 'double'), ('ship_name', 'string'), ('ship_address', 'string'), ('ship_city', 'string'), ('ship_region', 'string'), ('ship_postal_code', 'string'), ('ship_country', 'string')]
[('order_id', 'int'), ('product_id', 'int'), ('unit_price', 'double'), ('quantity', 'int'), ('discount', 'double')]


# Q14 — Nettoyage des chaînes
Dans df_customers, appliquer TRIM sur toutes les colonnes texte. Mettre contact_name en title case avec
initcap(). Mettre country en majuscules avec upper().

In [9]:
df_customers.dtypes

[('customer_id', 'string'),
 ('company_name', 'string'),
 ('contact_name', 'string'),
 ('contact_title', 'string'),
 ('address', 'string'),
 ('city', 'string'),
 ('region', 'string'),
 ('postal_code', 'string'),
 ('country', 'string'),
 ('phone', 'string'),
 ('fax', 'string')]

In [10]:
# Pour chaque colonne dont le type est String, on effectue un TRIM sur la colonne
for c in df_customers.columns:
    if df_customers.schema[c].dataType == StringType():
        df_customers = df_customers.withColumn(c, trim(col(c)))

# Transformations spécifiques
df_customers = df_customers.withColumn("contact_name", initcap(col("contact_name")));
df_customers = df_customers.withColumn("country", upper(col("country")));

df_customers.head(5)


[Row(customer_id='ALFKI', company_name='Alfreds Futterkiste', contact_name='Maria Anders', contact_title='Sales Representative', address='Obere Str. 57', city='Berlin', region=None, postal_code='12209', country='GERMANY', phone='030-0074321', fax='030-0076545'),
 Row(customer_id='ANATR', company_name='Ana Trujillo Emparedados y helados', contact_name='Ana Trujillo', contact_title='Owner', address='Avda. de la Constitución 2222', city='México D.F.', region=None, postal_code='05021', country='MEXICO', phone='(5) 555-4729', fax='(5) 555-3745'),
 Row(customer_id='ANTON', company_name='Antonio Moreno Taquería', contact_name='Antonio Moreno', contact_title='Owner', address='Mataderos  2312', city='México D.F.', region=None, postal_code='05023', country='MEXICO', phone='(5) 555-3932', fax=None),
 Row(customer_id='AROUT', company_name='Around the Horn', contact_name='Thomas Hardy', contact_title='Sales Representative', address='120 Hanover Sq.', city='London', region=None, postal_code='WA1 1DP

# Q15 — Renommer les colonnes
Dans df_order_details, renommer unit_price en prix_unitaire et quantity en quantite. Dans df_orders, renommer
ship_via en shipper_id.

In [11]:
col_renamed_orders_details = {"unit_price": "prix_unitaire", "quantity" : "quantite"}

# Renommer plusieurs colonnes a la fois
df_orders_details = df_orders_details.withColumnsRenamed(col_renamed_orders_details);

# Renommer une seule colonne
df_orders = df_orders.withColumnRenamed("ship_via", "shipper_id");

In [12]:
print(df_orders_details.columns)
print(df_orders.columns)

['order_id', 'product_id', 'prix_unitaire', 'quantite', 'discount']
['order_id', 'customer_id', 'employee_id', 'order_date', 'required_date', 'shipped_date', 'shipper_id', 'freight', 'ship_name', 'ship_address', 'ship_city', 'ship_region', 'ship_postal_code', 'ship_country']


# Q16 — Colonnes calculées
Dans df_order_details, ajouter une colonne sous_total = prix_unitaire * quantite * (1 - discount). Arrondir à 2
décimales avec round().

In [13]:
df_orders_details.head(5)

[Row(order_id=10248, product_id=11, prix_unitaire=14.0, quantite=12, discount=0.0),
 Row(order_id=10248, product_id=42, prix_unitaire=9.8, quantite=10, discount=0.0),
 Row(order_id=10248, product_id=72, prix_unitaire=34.8, quantite=5, discount=0.0),
 Row(order_id=10249, product_id=14, prix_unitaire=18.6, quantite=9, discount=0.0),
 Row(order_id=10249, product_id=51, prix_unitaire=42.4, quantite=40, discount=0.0)]

In [14]:
df_orders_details = df_orders_details.withColumn(
    "sous_total", 
    round(
        df_orders_details.prix_unitaire
        * df_orders_details.quantite
        * (1 - df_orders_details.discount)
    ,2));

In [15]:
df_orders_details.head(5)

[Row(order_id=10248, product_id=11, prix_unitaire=14.0, quantite=12, discount=0.0, sous_total=168.0),
 Row(order_id=10248, product_id=42, prix_unitaire=9.8, quantite=10, discount=0.0, sous_total=98.0),
 Row(order_id=10248, product_id=72, prix_unitaire=34.8, quantite=5, discount=0.0, sous_total=174.0),
 Row(order_id=10249, product_id=14, prix_unitaire=18.6, quantite=9, discount=0.0, sous_total=167.4),
 Row(order_id=10249, product_id=51, prix_unitaire=42.4, quantite=40, discount=0.0, sous_total=1696.0)]

# Q17 — Colonnes conditionnelles
Dans df_products, ajouter une colonne en_stock (True si units_in_stock > 0). Dans df_orders, ajouter une
colonne is_shipped (True si shipped_date n'est pas null).

In [16]:
df_products = df_products.withColumn("en_stock", df_products.units_in_stock > 0)
df_orders = df_orders.withColumn("is_shipped", df_orders.shipped_date.isNotNull())

In [17]:
df_products.head(5)

[Row(product_id=1, product_name='Chai', supplier_id=8, category_id=1, quantity_per_unit='10 boxes x 30 bags', unit_price=18.0, units_in_stock=39, units_on_order=0, reorder_level=10, discontinued=1, en_stock=True),
 Row(product_id=2, product_name='Chang', supplier_id=1, category_id=1, quantity_per_unit='24 - 12 oz bottles', unit_price=19.0, units_in_stock=17, units_on_order=40, reorder_level=25, discontinued=1, en_stock=True),
 Row(product_id=3, product_name='Aniseed Syrup', supplier_id=1, category_id=2, quantity_per_unit='12 - 550 ml bottles', unit_price=10.0, units_in_stock=13, units_on_order=70, reorder_level=25, discontinued=0, en_stock=True),
 Row(product_id=4, product_name="Chef Anton's Cajun Seasoning", supplier_id=2, category_id=2, quantity_per_unit='48 - 6 oz jars', unit_price=22.0, units_in_stock=53, units_on_order=0, reorder_level=0, discontinued=0, en_stock=True),
 Row(product_id=5, product_name="Chef Anton's Gumbo Mix", supplier_id=2, category_id=2, quantity_per_unit='36 bo

In [18]:
df_orders.head(5)

[Row(order_id=10248, customer_id='VINET', employee_id=5, order_date=datetime.date(1996, 7, 4), required_date=datetime.date(1996, 8, 1), shipped_date=datetime.date(1996, 7, 16), shipper_id=3, freight=32.38, ship_name='Vins et alcools Chevalier', ship_address="59 rue de l'Abbaye", ship_city='Reims', ship_region=None, ship_postal_code='51100', ship_country='France', is_shipped=True),
 Row(order_id=10249, customer_id='TOMSP', employee_id=6, order_date=datetime.date(1996, 7, 5), required_date=datetime.date(1996, 8, 16), shipped_date=datetime.date(1996, 7, 10), shipper_id=1, freight=11.61, ship_name='Toms Spezialitäten', ship_address='Luisenstr. 48', ship_city='Münster', ship_region=None, ship_postal_code='44087', ship_country='Germany', is_shipped=True),
 Row(order_id=10250, customer_id='HANAR', employee_id=4, order_date=datetime.date(1996, 7, 8), required_date=datetime.date(1996, 8, 5), shipped_date=datetime.date(1996, 7, 12), shipper_id=2, freight=65.83, ship_name='Hanari Carnes', ship_ad

# Q18 — Doublons
Vérifier s'il y a des doublons dans df_customers sur customer_id. Utiliser distinct() et count() pour comparer.
Supprimer les doublons si nécessaire.

In [19]:
print("Nombre de customer_id dans df_customers : ", df_customers.select("customer_id").count())
print("Nombre de customer_id differents dans df_customers : ", df_customers.select("customer_id").distinct().count())

Nombre de customer_id dans df_customers :  91
Nombre de customer_id differents dans df_customers :  91


Pas de doublons a supprimer

Commande pour enlever les doublons sur la colonne customer_id

df_customers = df_customers.dropDuplicates(["customer_id"])

# Q19 — Filtrage
Filtrer df_orders pour ne garder que les commandes de 1997. Filtrer df_products pour ne garder que les produits
en stock (units_in_stock > 0) et non discontinués.

In [ ]:
print("Nombre de commandes : ", df_orders.count())
#df_orders_1997 = df_orders.filter(df_orders.order_date.between("1997-01-01","1997-12-31"))
df_orders = df_orders.filter(df_orders.order_date.like("1997-%-%"))
print("Nombre de commandes de 1997 : ", df_orders.count())

Nombre de commandes :  809
Nombre de commandes de 1997 :  408


In [21]:
print("Nombre de produits : ", df_products.count())
df_products = df_products.filter((df_products.units_in_stock > 0) & (df_products.discontinued == 0))
print("Nombre de produits en stock et non discontinués : ", df_products.count())

Nombre de produits :  77
Nombre de produits en stock et non discontinués :  66


# Q20 — Sélection de colonnes
Dans df_employees, sélectionner uniquement : employee_id, first_name, last_name, title, hire_date, city,
country. Créer une colonne full_name = first_name + ' ' + last_name.

In [22]:
df_employees.columns

['employee_id',
 'last_name',
 'first_name',
 'title',
 'title_of_courtesy',
 'birth_date',
 'hire_date',
 'address',
 'city',
 'region',
 'postal_code',
 'country',
 'home_phone',
 'extension',
 'photo',
 'notes',
 'reports_to',
 'photo_path']

In [23]:
df_employees = df_employees.select("employee_id", "first_name", "last_name", "title", "hire_date", "city", "country") \
    .withColumn("full_name", concat(df_employees.first_name, lit(" "), df_employees.last_name));

df_employees.head(5)

[Row(employee_id=1, first_name='Nancy', last_name='Davolio', title='Sales Representative', hire_date=datetime.date(1992, 5, 1), city='Seattle', country='USA', full_name='Nancy Davolio'),
 Row(employee_id=2, first_name='Andrew', last_name='Fuller', title='Vice President, Sales', hire_date=datetime.date(1992, 8, 14), city='Tacoma', country='USA', full_name='Andrew Fuller'),
 Row(employee_id=3, first_name='Janet', last_name='Leverling', title='Sales Representative', hire_date=datetime.date(1992, 4, 1), city='Kirkland', country='USA', full_name='Janet Leverling'),
 Row(employee_id=4, first_name='Margaret', last_name='Peacock', title='Sales Representative', hire_date=datetime.date(1993, 5, 3), city='Redmond', country='USA', full_name='Margaret Peacock'),
 Row(employee_id=5, first_name='Steven', last_name='Buchanan', title='Sales Manager', hire_date=datetime.date(1993, 10, 17), city='London', country='UK', full_name='Steven Buchanan')]

# Bonus
Écrire les DataFrames nettoyés en Parquet : PATH = "/home/jovyan/data/tmp"

In [24]:
#Reassignation des DataFrames a la collection, car dans PySpark les df sont immuables.

df_collection["categories"] = df_categories
df_collection["clients"] = df_customers
df_collection["commandes"] = df_orders
df_collection["details_commandes"] = df_orders_details
df_collection["employees"] = df_employees
df_collection["fournisseurs"] = df_suppliers
df_collection["produits"] = df_products
df_collection["transporteurs"] = df_shippers

filePath = "../data/tmp/"

for name, df in df_collection.items():
    df.write.parquet(filePath + name, mode="overwrite");